# 01 — LLM, VLM, and Agent-or-Not

## Objectives

- Understand an LLM API call
- Inspect input and output tokens
- Understand temperature
- Generate structured JSON output
- Track token usage and estimated cost
- Compare text-only LLM and vision-language model usage
- Solve the same task using:
  1. Single prompt
  2. Fixed workflow
  3. Agent
- Compare cost, latency, accuracy, and predictability

In [1]:
import os
import json
import time

from dotenv import load_dotenv
from openai import OpenAI

In [2]:
load_dotenv()

api_key = os.getenv("OPENAI_API_KEY")

if not api_key:
    raise ValueError("OPENAI_API_KEY is not set.")

print("API key loaded successfully.")

API key loaded successfully.


In [4]:
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)

MODEL = "openai/gpt-oss-20b"

print("Client created.")
print("Model:", MODEL)

Client created.
Model: openai/gpt-oss-20b


In [5]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain what an AI agent is in two sentences."
        }
    ]
)

print(response.choices[0].message.content)

An AI agent is a software system that perceives its environment through sensors or data inputs and takes actions through actuators or output channels to accomplish a set of objectives or maximize a reward function. By continually processing feedback, it adapts its behavior—often via learning algorithms—so its decisions become more effective over time.


In [6]:
print(response)

ChatCompletion(id='chatcmpl-183e3049-839b-4a38-86d9-2ed41610ab97', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='An AI agent is a software system that perceives its environment through sensors or data inputs and takes actions through actuators or output channels to accomplish a set of objectives or maximize a reward function. By continually processing feedback, it adapts its behavior—often via learning algorithms—so its decisions become more effective over time.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='We need to answer: "Explain what an AI agent is in two sentences." So we need concise two sentences explanation. Should be clear: an AI agent is a program that perceives environment, acts to achieve goals, learns. So two sentences.'))], created=1789982565, model='openai/gpt-oss-20b', object='chat.completion', metadata=None, moderation=None, service_tier='on_demand

In [7]:
usage = response.usage

print("Prompt tokens:", usage.prompt_tokens)
print("Completion tokens:", usage.completion_tokens)
print("Total tokens:", usage.total_tokens)

Prompt tokens: 81
Completion tokens: 123
Total tokens: 204


In [8]:
start_time = time.perf_counter()

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Explain what an AI agent is in two sentences."
        }
    ]
)

end_time = time.perf_counter()

latency = end_time - start_time

print("Response:")
print(response.choices[0].message.content)

print("\nLatency:", round(latency, 3), "seconds")
print("Prompt tokens:", response.usage.prompt_tokens)
print("Completion tokens:", response.usage.completion_tokens)
print("Total tokens:", response.usage.total_tokens)

Response:
An AI agent is a software entity that perceives its environment through sensors, processes that information, and makes decisions to achieve a goal. It then acts upon the environment via effectors, continually learning and adapting its behavior to improve performance.

Latency: 0.617 seconds
Prompt tokens: 81
Completion tokens: 102
Total tokens: 183


In [9]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Give me one short definition of RAG."
        }
    ],
    temperature=0
)

print(response.choices[0].message.content)

RAG: Retrieval‑Augmented Generation – a technique that fetches relevant external documents to inform and improve a language model’s output.


In [10]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Give me one short definition of RAG."
        }
    ],
    temperature=1
)

print(response.choices[0].message.content)

**RAG**: A color‑coded status system—**Red** for critical, **Amber** (or yellow) for warning, and **Green** for normal or safe.


In [11]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "Return only valid JSON."
        },
        {
            "role": "user",
            "content": """
Extract the following information:

Scheme: Student Support Scheme
Eligibility: Students meeting the specified income criteria
Documents: Aadhaar, income certificate, student ID
"""
        }
    ],
    response_format={"type": "json_object"}
)

result = response.choices[0].message.content

print(result)

{"Scheme":"Student Support Scheme","Eligibility":"Students meeting the specified income criteria","Documents":["Aadhaar","income certificate","student ID"]}


In [12]:
data = json.loads(result)

print(data)
print(type(data))

{'Scheme': 'Student Support Scheme', 'Eligibility': 'Students meeting the specified income criteria', 'Documents': ['Aadhaar', 'income certificate', 'student ID']}
<class 'dict'>


In [13]:
def calculate_cost(
    prompt_tokens,
    completion_tokens,
    input_price_per_million,
    output_price_per_million
):
    input_cost = (
        prompt_tokens / 1_000_000
    ) * input_price_per_million

    output_cost = (
        completion_tokens / 1_000_000
    ) * output_price_per_million

    return input_cost + output_cost

In [14]:
def print_usage(response):
    usage = response.usage

    print("Prompt tokens:", usage.prompt_tokens)
    print("Completion tokens:", usage.completion_tokens)
    print("Total tokens:", usage.total_tokens)

In [15]:
print_usage(response)

Prompt tokens: 139
Completion tokens: 158
Total tokens: 297


In [19]:
INPUT_PRICE_PER_MILLION = 2.0
OUTPUT_PRICE_PER_MILLION = 3.0

In [20]:
cost = calculate_cost(
    response.usage.prompt_tokens,
    response.usage.completion_tokens,
    INPUT_PRICE_PER_MILLION,
    OUTPUT_PRICE_PER_MILLION
)

print(f"Estimated cost: ${cost:.8f}")

Estimated cost: $0.00075200


In [21]:
from pathlib import Path
import base64

image_path = Path("chart.jpg")

with open(image_path, "rb") as image_file:
    image_base64 = base64.b64encode(
        image_file.read()
    ).decode("utf-8")

print("Image loaded:", image_path)

Image loaded: chart.jpg


In [26]:
VISION_MODEL = "qwen/qwen3.8-27b"

In [27]:
print(VISION_MODEL)

qwen/qwen3.8-27b


In [28]:
start_time = time.perf_counter()

response = client.chat.completions.create(
    model=VISION_MODEL,
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": "What is the highest value shown in this chart?"
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/png;base64,{image_base64}"
                    }
                }
            ]
        }
    ],
    temperature=1,
    max_completion_tokens=512
)

end_time = time.perf_counter()

print("Answer:")
print(response.choices[0].message.content)

print("\nLatency:", round(end_time - start_time, 3), "seconds")

if response.usage:
    print("Prompt tokens:", response.usage.prompt_tokens)
    print("Completion tokens:", response.usage.completion_tokens)
    print("Total tokens:", response.usage.total_tokens)

Answer:
Looking at the bar chart, we can observe the following:

- The **y-axis** shows values from 0 to 20 in increments of 2.
- Each bar corresponds to a date (1/16 through 7/16).
- The **tallest bar** is the **yellow bar labeled “5/16”**, which reaches the top grid line marked **20**.

Additionally, the zoomed-in views confirm that the yellow bar (5/16) extends to the 20 mark on the y-axis.

---

✅ Therefore, the **highest value shown in the chart is 20**.

Latency: 1.682 seconds
Prompt tokens: 1816
Completion tokens: 136
Total tokens: 1952


## VLM vs Text-only LLM

### Text-only

- Receives text
- Cannot directly inspect the chart image
- Requires the chart information to be converted into text

### VLM

- Receives image + question
- Can analyze visual information
- Useful for charts, screenshots, forms, diagrams, and images

### Typical VLM failure cases

- Tiny text
- Blurry images
- Complex charts
- Exact counting
- Fine-grained visual details
- Incorrect interpretation of spatial relationships

In [29]:
task_prompt = """
Determine whether the following student satisfies the eligibility rules.

Rules:
- Age must be at least 18.
- Annual income must be no more than ₹300,000.

Student:
- Age: 22
- Annual income: ₹250,000

Return:
1. Eligible or Not Eligible
2. A short explanation
"""

start_time = time.perf_counter()

response_single = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": task_prompt
        }
    ],
    temperature=0
)

end_time = time.perf_counter()

single_latency = end_time - start_time

print(response_single.choices[0].message.content)
print("Latency:", round(single_latency, 3), "seconds")
print("Tokens:", response_single.usage.total_tokens)

1. **Eligible**  
2. The student is 22 years old, which meets the minimum age requirement of 18, and earns ₹250,000 annually, which is below the maximum allowed income of ₹300,000. Hence, the student satisfies both eligibility criteria.
Latency: 0.569 seconds
Tokens: 257


In [30]:
age = 22
income = 250_000

age_valid = age >= 18
income_valid = income <= 300_000

eligible = age_valid and income_valid

print("Age valid:", age_valid)
print("Income valid:", income_valid)
print("Eligible:", eligible)

Age valid: True
Income valid: True
Eligible: True


In [31]:
workflow_prompt = f"""
Explain the eligibility result clearly.

Age: {age}
Age requirement satisfied: {age_valid}

Annual income: ₹{income}
Income requirement satisfied: {income_valid}

Final eligibility result: {eligible}
"""

start_time = time.perf_counter()

response_workflow = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": workflow_prompt
        }
    ],
    temperature=0
)

end_time = time.perf_counter()

workflow_latency = end_time - start_time

print(response_workflow.choices[0].message.content)
print("Latency:", round(workflow_latency, 3), "seconds")
print("Tokens:", response_workflow.usage.total_tokens)

**Eligibility Summary**

| Criterion | Value | Requirement | Met? |
|-----------|-------|-------------|------|
| Age | 22 years | ≥ 18 years (typical minimum) | ✅ |
| Annual Income | ₹250,000 | ≥ ₹200,000 (typical minimum) | ✅ |

**Why the final result is “True”**

1. **Age Check** – The applicant is 22 years old, which is above the minimum age threshold (usually 18 years).  
2. **Income Check** – The applicant’s annual income of ₹250,000 exceeds the required minimum of ₹200,000.  

Since **both** conditions are satisfied, the system flags the applicant as eligible, yielding a final eligibility result of **True**. If either the age or income had fallen below its respective threshold, the final result would have been **False**.
Latency: 1.18 seconds
Tokens: 519


In [32]:
def check_eligibility(age: int, income: float) -> dict:
    age_valid = age >= 18
    income_valid = income <= 300_000

    return {
        "age_valid": age_valid,
        "income_valid": income_valid,
        "eligible": age_valid and income_valid
    }

In [33]:
result = check_eligibility(22, 250_000)

print(result)

{'age_valid': True, 'income_valid': True, 'eligible': True}


In [34]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "check_eligibility",
            "description": "Check student eligibility based on age and income.",
            "parameters": {
                "type": "object",
                "properties": {
                    "age": {
                        "type": "integer"
                    },
                    "income": {
                        "type": "number"
                    }
                },
                "required": ["age", "income"]
            }
        }
    }
]

In [35]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": """
            Determine whether a student with age 22
            and annual income ₹250,000 is eligible.
            """
        }
    ],
    tools=tools,
    tool_choice="auto"
)

In [36]:
print(response.choices[0].message)

ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='fc_b40347ac-3dcf-48e2-a496-50fa1fd24250', function=Function(arguments='{"age":22,"income":250000}', name='check_eligibility'), type='function')], reasoning='We need to use the function check_eligibility with age=22, income=250000.')


In [37]:
time.perf_counter()

18101.9468556

# Final Comparison

We solved the same eligibility task using three architectures:

## 1. Single Prompt

The LLM received the complete problem and generated the answer.

## 2. Fixed Workflow

Python performed the deterministic eligibility calculation and the LLM generated the explanation.

## 3. Agent

The LLM was given an eligibility tool and could decide to call the tool before producing the answer.

### Metrics

For each approach we recorded:

- Latency
- Prompt tokens
- Completion tokens
- Total tokens
- Estimated cost
- Accuracy on a small evaluation set
- Predictability

In [38]:
results = [
    {
        "approach": "Single Prompt",
        "latency_seconds": single_latency,
        "tokens": response_single.usage.total_tokens,
    },
    {
        "approach": "Fixed Workflow",
        "latency_seconds": workflow_latency,
        "tokens": response_workflow.usage.total_tokens,
    },
]

for result in results:
    print(result)

{'approach': 'Single Prompt', 'latency_seconds': 0.5687980000002426, 'tokens': 257}
{'approach': 'Fixed Workflow', 'latency_seconds': 1.1798096999991685, 'tokens': 519}
